# 04 - Data Preparation: Undersampling & Oversampling Exploration
This notebook performs an exploration of the data undersampling and oversampling techniques. It loads the processed dataset from `data/processed/` folder and shows the effect of the different strategies.

## Import libraries and settings

In [ ]:
from __future__ import annotations

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from imblearn.base import BaseSampler
from typing import Sequence, Union
from pathlib import Path

from fraud_dynamic_ensemble.config import PROCESSED_DATA_DIR, PROCESSED_FILENAME, FIGURES_DIR, RANDOM_STATE

In [ ]:
input_path: Path = PROCESSED_DATA_DIR / PROCESSED_FILENAME
print(f"Loading dataset at path:\n\t{input_path}")

In [ ]:
FIGURES_DATA_IMBALANCE_DIR = FIGURES_DIR / "DU_data_undersampling_oversampling"
FIGURES_DATA_IMBALANCE_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
pd.set_option("display.max_columns", None)
plt.rcParams.update({"font.size": 16})
sns.set_style("whitegrid")
sns.set_palette("tab10")

## Data Loading
We begin by loading the interim credit card fraud dataset. This is the cleaned file after running data-cleaning phase. It contains cleaned anonymized PCA components (`V1`–`V28`), `Time`, `Amount`, and the `Class` label.

In [ ]:
df = pd.read_csv(input_path, header=0, sep=",")
print(f"Dataset dimension\n"
      f"\t-Number of rows = {df.shape[0]}\n"
      f"\t-Number of cols = {df.shape[1]}")

## Basic Overview

In [ ]:
df.head(10)

In [ ]:
df.tail(10)

In [ ]:
columns_name = df.drop(columns="Class").columns.tolist()
columns_name

In [ ]:
X = df.drop(columns="Class").to_numpy()
y = df["Class"].to_numpy()

##  Resampling Visualization Function

In [ ]:
def visualize_resampling_effect(
    X: Union[np.ndarray, Sequence[Sequence[float]]],
    y: Union[np.ndarray, Sequence],
    resampler: BaseSampler,
    column_names: list[str],
    title: str,
    base_path: Union[str, Path],
) -> None:
    """
    Visualize the effect of a resampling method using an 8-panel comparison.

    The function resamples (X, y) with the provided ``resampler`` (from imbalanced-learn),
    concatenates BEFORE/AFTER DataFrames, and renders a 2×4 grid:

        Row 1 (BEFORE):  [Scatter V1×V2] [KDE V1] [KDE V2] [Stripplot V1&V2]
        Row 2 (AFTER):   [Scatter V1×V2] [KDE V1] [KDE V2] [Stripplot V1&V2]

    It expects feature names to include ``"V1"`` and ``"V2"`` for the bivariate panels.
    If not found, it falls back to the first two columns in ``column_names``.

    Parameters
    ----------
    X : array-like of shape (n_samples, n_features)
        Original feature matrix.
    y : array-like of shape (n_samples,)
        Class labels.
    resampler : imblearn.base.BaseSampler
        Fitted or unfitted resampler (e.g., RandomUnderSampler, SMOTE, etc.).
    column_names : list
        Feature names for DataFrame construction. Should contain at least two names.
    title : str
        Title prefix used in subplot titles and output filename.
    base_path : str or pathlib.Path
        Directory to save the resulting figure. Created if missing.

    Returns
    -------
    None
        Saves a PNG figure and prints class counts before/after.

    Notes
    -----
    - KDEs are per-feature, per-class, with ``fill=True`` and ``common_norm=False``.
    - The stripplot replaces the older violin in your original version, showing
      class-wise scatter of V1/V2 values (dodge=True).
    - Axis limits for scatter/KDE are synchronized across BEFORE/AFTER for fair comparison.

    Examples
    --------
    >>> from imblearn.under_sampling import RandomUnderSampler
    >>> visualize_resampling_effect(
    ...     X, y,
    ...     resampler=RandomUnderSampler(random_state=42),
    ...     column_names=["V1","V2", "V3", ...],
    ...     title="RandomUnderSampler",
    ...     base_path="reports/figures/DP_sampling_checks"
    ... )
    """
    # Resample
    X = np.asarray(X)
    y = np.asarray(y)
    X_res, y_res = resampler.fit_resample(X, y)

    # Class counts BEFORE/AFTER
    for label, (uniq, freq) in [
        ("Before", np.unique(y, return_counts=True)),
        ("After",  np.unique(y_res, return_counts=True)),
    ]:
        print(f"Class balance {label} {title}:\n\t{dict(zip(uniq, freq))}")

    # Build DataFrames and concat with Stage flag
    df_before = pd.DataFrame(X, columns=column_names)
    df_before["Class"] = y
    df_before["Stage"] = "Before"

    df_after = pd.DataFrame(X_res, columns=column_names)
    df_after["Class"] = y_res
    df_after["Stage"] = "After"

    df_all = pd.concat([df_before, df_after], axis=0, ignore_index=True)

    # Resolve the two PCs to use (prefer 'V1','V2'; else first two columns)
    if {"V1", "V2"}.issubset(df_all.columns):
        f1, f2 = "V1", "V2"
    else:
        if len(column_names) < 2:
            raise ValueError("Need at least two feature names to plot (got fewer than 2).")
        f1, f2 = column_names[0], column_names[1]

    # Prepare figure & axes: 2 rows × 4 cols
    fig, axes = plt.subplots(2, 4, figsize=(28, 12))
    (ax_s_b, ax_k1_b, ax_k2_b, ax_str_b), (ax_s_a, ax_k1_a, ax_k2_a, ax_str_a) = axes

    # Helper: plot a row (stage subset)
    def _plot_row(subdf: pd.DataFrame, stage_label: str,
                  ax_scatter, ax_kde1, ax_kde2, ax_strip) -> None:
        # 1) Scatter f1×f2
        sns.scatterplot(
            data=subdf, x=f1, y=f2, hue="Class", ax=ax_scatter,
            alpha=0.6, palette="tab10", edgecolor=None, s=12, linewidth=0
        )
        ax_scatter.set_title(f"{title} – Scatter {stage_label}")
        ax_scatter.legend(title="Class")

        # 2) KDE for f1
        sns.kdeplot(
            data=subdf, x=f1, hue="Class", fill=True, common_norm=False,
            alpha=0.5, ax=ax_kde1, palette="tab10"
        )
        ax_kde1.set_title(f"{title} – KDE {f1} {stage_label}")

        # 3) KDE for f2
        sns.kdeplot(
            data=subdf, x=f2, hue="Class", fill=True, common_norm=False,
            alpha=0.5, ax=ax_kde2, palette="tab10"
        )
        ax_kde2.set_title(f"{title} – KDE {f2} {stage_label}")

        # 4) Stripplot for f1 & f2 by class
        melted = subdf.melt(
            id_vars="Class",
            value_vars=[f1, f2],
            var_name="Feature",
            value_name="Value",
        )
        sns.stripplot(
            data=melted, x="Feature", y="Value", hue="Class",
            dodge=True, alpha=0.35, ax=ax_strip, palette="tab10"
        )
        ax_strip.set_title(f"{title} – Stripplot {stage_label}")
        ax_strip.legend(title="Class", loc="best")

    # Plot BEFORE row
    _plot_row(df_all[df_all["Stage"] == "Before"], "Before", ax_s_b, ax_k1_b, ax_k2_b, ax_str_b)
    # Plot AFTER row
    _plot_row(df_all[df_all["Stage"] == "After"],  "After",  ax_s_a, ax_k1_a, ax_k2_a, ax_str_a)

    # Layout and save
    plt.tight_layout()
    safe_title = title.replace(" ", "_")
    fig_path = base_path / f"data_imbalance_{safe_title}.png"
    plt.savefig(fig_path, dpi=300)
    plt.show()

## Undersampling Techniques

We will explore the following resampling methods:

### Prototype generation:
- ClusterCentroids

### Prototype selection:
- RandomUnderSampler
- NearMiss_1
- NearMiss_2
- NearMiss_3
- TomekLinks
- EditedNearestNeighbours
- RepeatedEditedNearestNeighbours
- AllKNN
- CondensedNearestNeighbour
- OneSidedSelection
- NeighbourhoodCleaningRule
- InstanceHardnessThreshold

For each method, we visualize the effect on the feature `V1` and `V2`.

### Prototype generation

Given an original data set $S$, prototype generation algorithms will generate a new set $S'$ where |$S'$| < |$S$| and $S' \not \subset S$.

In other words, prototype generation techniques will reduce the number of samples in the targeted classes, but the remaining samples are generated — and not selected — from the original set.

#### `ClusterCentroids`
ClusterCentroids makes use of K-means to reduce the number of samples. Therefore, each class will be synthesized with the centroids of the K-means method instead of the original samples.

ClusterCentroids offers an efficient way to represent the data cluster with a reduced number of samples. Keep in mind that this method requires that your data are grouped into clusters. In addition, the number of centroids should be set such that the under-sampled clusters are representative of the original one.

In [ ]:
from imblearn.under_sampling import ClusterCentroids

visualize_resampling_effect(X, y,
                            ClusterCentroids(random_state=RANDOM_STATE),
                            columns_name,
                            "ClusterCentroids",
                            FIGURES_DATA_IMBALANCE_DIR)

### Prototype selection
Prototype selection algorithms will select samples from the original set $S$, generating a dataset $S'$, where $S'$| < |$S$| and $S' \subset S$. In other words, $S'$ is a subset of $S$.

Prototype selection algorithms can be divided into two groups: (i) controlled under-sampling techniques and (ii) cleaning under-sampling techniques.

*Controlled under-sampling* methods reduce the number of observations in the majority class or classes to an arbitrary number of samples specified by the user. Typically, they reduce the number of observations to the number of samples observed in the minority class.
Methods in this class are *RandomUnderSampler* and *NearMiss*.


In contrast, *cleaning under-sampling* techniques “clean” the feature space by removing either “noisy” or “too easy to classify” observations, depending on the method. The final number of observations in each class varies with the cleaning method and can’t be specified by the user.
Methods in this class are *TomekLinks*, *EditedNearestNeighbours*, *RepeatedEditedNearestNeighbours*, *AllKNN*, *CondensedNearestNeighbour*, *OneSidedSelection* and *InstanceHardnessThreshold*.

#### `RandomUnderSampler`
RandomUnderSampler is a fast and easy way to balance the data by randomly selecting a subset of data for the targeted classes.

RandomUnderSampler allows bootstrapping the data by setting replacement to True. When there are multiple classes, each targeted class is under-sampled independently.

In [ ]:
from imblearn.under_sampling import RandomUnderSampler

visualize_resampling_effect(X, y,
                            RandomUnderSampler(random_state=RANDOM_STATE),
                            columns_name,
                            "RandomUnderSampler",
                            FIGURES_DATA_IMBALANCE_DIR)

#### `NearMiss1`, `NearMiss2`, `NearMiss3`
Keeps majority samples that are closest (or most informative) w.r.t. minority points, using different neighbor rules:

- v1: select majority samples with the smallest average distance to the k nearest minority neighbors.

- v2: select majority samples with the smallest average distance to the k farthest minority neighbors.

- v3: for each minority sample, keep its m nearest majority neighbors.

In [ ]:
from imblearn.under_sampling import NearMiss

visualize_resampling_effect(
    X, y,
    NearMiss(version=1),
    columns_name,
    "NearMiss-1",
    FIGURES_DATA_IMBALANCE_DIR,
)

In [ ]:
visualize_resampling_effect(
    X, y,
    NearMiss(version=2),
    columns_name,
    "NearMiss-2",
    FIGURES_DATA_IMBALANCE_DIR,
)

In [ ]:
visualize_resampling_effect(
    X, y,
    NearMiss(version=3),
    columns_name,
    "NearMiss-3",
    FIGURES_DATA_IMBALANCE_DIR,
)

#### `TomekLinks`

Cleans the class boundary by removing samples that form Tomek links (mutual nearest neighbors with different labels). Typically drops the majority member of each link.

In [ ]:
from imblearn.under_sampling import TomekLinks

visualize_resampling_effect(
    X, y,
    TomekLinks(),
    columns_name,
    "TomekLinks",
    FIGURES_DATA_IMBALANCE_DIR,
)

#### `EditedNearestNeighbours (ENN)`

Removes samples whose label disagrees with the majority vote among their k nearest neighbors (default k=3), smoothing decision boundaries.

In [ ]:
from imblearn.under_sampling import EditedNearestNeighbours

visualize_resampling_effect(
    X, y,
    EditedNearestNeighbours(),  # k=3 by default
    columns_name,
    "EditedNearestNeighbours",
    FIGURES_DATA_IMBALANCE_DIR,
)

#### `RepeatedEditedNearestNeighbours (RENN)`

Applies ENN iteratively (until convergence or a max number of passes), producing stronger cleaning than a single ENN pass.

In [ ]:
from imblearn.under_sampling import RepeatedEditedNearestNeighbours

visualize_resampling_effect(
    X, y,
    RepeatedEditedNearestNeighbours(),  # repeats ENN until no change / max_iter
    columns_name,
    "RepeatedEditedNearestNeighbours",
    FIGURES_DATA_IMBALANCE_DIR,
)

#### `AllKNN`

Runs ENN for increasing k values (e.g., 1→…→k_max). This progressively strengthens the editing as neighborhood size grows.

In [ ]:
from imblearn.under_sampling import AllKNN

visualize_resampling_effect(
    X, y,
    AllKNN(),  # progressively increases k
    columns_name,
    "AllKNN",
    FIGURES_DATA_IMBALANCE_DIR,
)

#### `CondensedNearestNeighbour (CNN)`

Builds a small consistent subset of samples that preserves the 1-NN decision boundary; it removes redundant majority points far from the boundary.

In [ ]:
from imblearn.under_sampling import CondensedNearestNeighbour

visualize_resampling_effect(
    X, y,
    CondensedNearestNeighbour(random_state=RANDOM_STATE),
    columns_name,
    "CondensedNearestNeighbour",
    FIGURES_DATA_IMBALANCE_DIR,
)

#### `OneSidedSelection (OSS)`

Combines Tomek links removal with a condensation step to keep informative majority examples and discard noisy/redundant ones in a single procedure.

In [ ]:
from imblearn.under_sampling import OneSidedSelection

visualize_resampling_effect(
    X, y,
    OneSidedSelection(random_state=RANDOM_STATE),
    columns_name,
    "OneSidedSelection",
    FIGURES_DATA_IMBALANCE_DIR,
)

#### `NeighbourhoodCleaningRule (NCR)`

Uses ENN-like checks to remove misclassified majority samples in the neighborhood of minority points, reducing borderline noise without heavy undersampling.

In [ ]:
from imblearn.under_sampling import NeighbourhoodCleaningRule

visualize_resampling_effect(
    X, y,
    NeighbourhoodCleaningRule(),  # focuses on cleaning majority around minority
    columns_name,
    "NeighbourhoodCleaningRule",
    FIGURES_DATA_IMBALANCE_DIR,
)

#### `InstanceHardnessThreshold (IHT)`

Estimates each sample’s “hardness” (likelihood of being misclassified by a base estimator) and removes easy majority cases first, keeping challenging/ informative ones.

In [ ]:
from imblearn.under_sampling import InstanceHardnessThreshold
from sklearn.linear_model import LogisticRegression

visualize_resampling_effect(
    X, y,
    InstanceHardnessThreshold(
        estimator=LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)
    ),
    columns_name,
    "InstanceHardnessThreshold",
    FIGURES_DATA_IMBALANCE_DIR,
)


## Oversampling Techniques

We will explore the following resampling methods:

### Basic over-sampling:
- RandomOverSampler

### SMOTE algorithms:
- SMOTE
- ADASYN
- BorderlineSMOTE
- KMeansSMOTE
- SVMSMOTE

For each method, we visualize the effect on the feature `V1` and `V2`.



#### `RandomOverSampler`

Creates a balanced set by randomly duplicating minority samples until the target ratio is reached. Fast baseline; may increase overfitting due to duplicate rows.

In [ ]:
from imblearn.over_sampling import RandomOverSampler

visualize_resampling_effect(
    X, y,
    RandomOverSampler(random_state=RANDOM_STATE),
    columns_name,
    "RandomOverSampler",
    FIGURES_DATA_IMBALANCE_DIR,
)

#### `SMOTE`

Generates synthetic minority samples by interpolating between each minority point and its k nearest minority neighbors (default k_neighbors=5). Works on continuous features and assumes a meaningful distance metric (usually scale features in modeling).

In [ ]:
from imblearn.over_sampling import SMOTE

visualize_resampling_effect(
    X, y,
    SMOTE(random_state=RANDOM_STATE, k_neighbors=5),
    columns_name,
    "SMOTE",
    FIGURES_DATA_IMBALANCE_DIR,
)

#### `ADASYN`

Adaptive synthetic sampling: like SMOTE but focuses more on harder-to-learn minority samples (regions where minority density is low), creating more synthetic points there. Can increase boundary noise; monitor results.

In [ ]:
from imblearn.over_sampling import ADASYN

visualize_resampling_effect(
    X, y,
    ADASYN(random_state=RANDOM_STATE, n_neighbors=5),
    columns_name,
    "ADASYN",
    FIGURES_DATA_IMBALANCE_DIR,
)

#### `BorderlineSMOTE`

Generates synthetic samples near the decision boundary.

- Type 1: synthesize only around borderline minority samples.

- Type 2: also considers samples slightly inside majority regions (more aggressive).

In [ ]:
from imblearn.over_sampling import BorderlineSMOTE

visualize_resampling_effect(
    X, y,
    BorderlineSMOTE(kind="borderline-1", random_state=RANDOM_STATE, k_neighbors=5),
    columns_name,
    "BorderlineSMOTE-1",
    FIGURES_DATA_IMBALANCE_DIR,
)

In [ ]:
visualize_resampling_effect(
    X, y,
    BorderlineSMOTE(kind="borderline-2", random_state=RANDOM_STATE, k_neighbors=5),
    columns_name,
    "BorderlineSMOTE-2",
    FIGURES_DATA_IMBALANCE_DIR,
)

#### `KMeansSMOTE`

First clusters the minority class using k-means, then applies SMOTE within clusters, allocating more samples to sparse clusters. Helps reduce over-generalization and mode collapse.

In [ ]:
from imblearn.over_sampling import KMeansSMOTE

visualize_resampling_effect(
    X, y,
    KMeansSMOTE(random_state=RANDOM_STATE, k_neighbors=5),
    columns_name,
    "KMeansSMOTE",
    FIGURES_DATA_IMBALANCE_DIR,
)

#### `SVMSMOTE`

Uses an SVM to identify support vectors/borderline minority samples, then synthesizes points around those regions. Focused on the boundary; useful when classes are tightly intertwined.

In [ ]:
from imblearn.over_sampling import SVMSMOTE

visualize_resampling_effect(
    X, y,
    SVMSMOTE(random_state=RANDOM_STATE, k_neighbors=5),
    columns_name,
    "SVMSMOTE",
    FIGURES_DATA_IMBALANCE_DIR,
)

## Combination of over- and under-sampling

We will explore the following resampling methods:

- SMOTEENN
- SMOTETomek

For each method, we visualize the effect on the feature `V1` and `V2`.



#### `SMOTEENN`

First applies SMOTE (synthetic oversampling of the minority), then uses Edited Nearest Neighbours (ENN) to clean ambiguous/noisy samples (typically near the decision boundary) from both classes. You want a balanced set and to prune overlapped regions after oversampling.

Notes: If your minority is tiny, lower k_neighbors in SMOTE and n_neighbors in ENN to avoid neighbor errors.

In [ ]:
from imblearn.combine import SMOTEENN
from imblearn.over_sampling import SMOTE
from imblearn.under_sampling import EditedNearestNeighbours

resampler = SMOTEENN(
    random_state=RANDOM_STATE,
    smote=SMOTE(random_state=RANDOM_STATE, k_neighbors=5),
    enn=EditedNearestNeighbours(n_neighbors=3)
)

visualize_resampling_effect(
    X, y,
    resampler,
    columns_name,
    "SMOTEENN",
    FIGURES_DATA_IMBALANCE_DIR,
)

#### `SMOTETomek`

Applies SMOTE to oversample the minority, then removes Tomek links (pairs of nearest neighbors from opposite classes) to sharpen the class boundary. You want a cleaner margin after oversampling, but less aggressive cleaning than ENN.

Notes: Tomek removal can slightly reduce both classes along the boundary; tune k_neighbors in SMOTE as needed.

In [ ]:
from imblearn.combine import SMOTETomek
from imblearn.over_sampling import SMOTE
from imblearn.under_sampling import TomekLinks

resampler = SMOTETomek(
    random_state=RANDOM_STATE,
    smote=SMOTE(random_state=RANDOM_STATE, k_neighbors=5),
    tomek=TomekLinks()
)

visualize_resampling_effect(
    X, y,
    resampler,
    columns_name,
    "SMOTETomek",
    FIGURES_DATA_IMBALANCE_DIR,
)